# Best-fit Probabilistic Distribution of Time Headway, per Pair

`Pair` showed a large, significant association with `Time_Headway`, so the headway
distribution is modelled **separately for each of the 8 pairs**.

**Procedure**
1. Fit 10 candidate distributions to each pair's headways by maximum likelihood.
2. Rank fits by **AIC** (primary) with **BIC**, log-likelihood and the KS statistic reported.
3. Select the best distribution per pair; write results + fitted-PDF overlay charts to Excel.

**Candidate distributions** (all common in headway literature): lognormal, gamma,
Weibull, inverse Gaussian, exponential, log-logistic (Fisk), Pearson III,
generalized gamma, GEV, Rayleigh.

*Note on graphics:* because the Application Control policy on this machine blocks
matplotlib's compiled DLLs, the fitted-PDF overlays are drawn as **native Excel
charts** embedded in the output workbook (in the `Tables` folder) rather than PNGs.

In [1]:
# --- Cell 1: Imports and paths ---
import os, warnings
import numpy as np
import pandas as pd
from scipy import stats
from openpyxl import load_workbook
from openpyxl.chart import ScatterChart, Reference, Series

BASE      = r"D:\Headway"
DATA_PATH = os.path.join(BASE, "data2.xlsx")
TABLES    = os.path.join(BASE, "Tables")
os.makedirs(TABLES, exist_ok=True)

OUTCOME   = "Time_Headway"
GROUP     = "Pair"
SMALL_N   = 50   # pairs below this get a caution flag

CANDIDATES = ["lognorm", "gamma", "weibull_min", "invgauss", "expon",
              "fisk", "pearson3", "gengamma", "genextreme", "rayleigh"]

In [2]:
# --- Cell 2: Load data and show headway counts per pair ---
df = pd.read_excel(DATA_PATH)
counts = df.groupby(GROUP)[OUTCOME].size().sort_values(ascending=False)
print("Headway observations per pair:")
print(counts.to_string())

Headway observations per pair:
Pair
BTW_following_4W        250
BTW_following_MT_3W     186
BTW_following_NMT_3W    160
PR_following_MT_3W      104
BTW_following_2W         73
PR_following_NMT_3W      59
PR_following_4W          43
PR_following_2W          23


In [3]:
# --- Cell 3: Fit one distribution to a data vector ---
def fit_one(name, data):
    """MLE-fit a scipy distribution; return fit metrics or None on failure."""
    dist = getattr(stats, name)
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            params = dist.fit(data)
        ll = np.sum(dist.logpdf(data, *params))
        if not np.isfinite(ll):
            return None
        k, n = len(params), len(data)
        aic = 2 * k - 2 * ll
        bic = k * np.log(n) - 2 * ll
        ks_stat, ks_p = stats.kstest(data, name, args=params)
        shapes = dist.shapes
        labels = ([s.strip() for s in shapes.split(",")] if shapes else []) + ["loc", "scale"]
        param_str = ", ".join(f"{lab}={val:.4f}" for lab, val in zip(labels, params))
        return dict(Distribution=name, n_params=k, LogLik=ll,
                    AIC=aic, BIC=bic, KS_stat=ks_stat, KS_p=ks_p,
                    Params=param_str, _raw=params)
    except Exception:
        return None

In [4]:
# --- Cell 4: Fit every candidate to every pair ---
all_rows   = []                       # long table: pair x distribution
best_rows  = []                       # one best row per pair
best_store = {}                       # pair -> (dist_name, raw_params, data) for charts

for pair, g in df.groupby(GROUP):
    data = g[OUTCOME].dropna().values
    n = len(data)
    fits = [f for f in (fit_one(name, data) for name in CANDIDATES) if f is not None]
    fits.sort(key=lambda d: d["AIC"])           # best (lowest AIC) first

    for rank, f in enumerate(fits, 1):
        all_rows.append({
            "Pair": pair, "N": n, "Rank_by_AIC": rank,
            "Distribution": f["Distribution"], "n_params": f["n_params"],
            "LogLik": round(f["LogLik"], 2), "AIC": round(f["AIC"], 2),
            "BIC": round(f["BIC"], 2), "KS_stat": round(f["KS_stat"], 4),
            "KS_p": round(f["KS_p"], 4), "Params": f["Params"],
        })

    b = fits[0]
    best_store[pair] = (b["Distribution"], b["_raw"], data)
    best_rows.append({
        "Pair": pair, "N": n,
        "Best_Distribution": b["Distribution"],
        "Parameters": b["Params"],
        "AIC": round(b["AIC"], 2), "BIC": round(b["BIC"], 2),
        "LogLik": round(b["LogLik"], 2),
        "KS_stat": round(b["KS_stat"], 4), "KS_p": round(b["KS_p"], 4),
        "Sample_flag": "caution (small n)" if n < SMALL_N else "ok",
    })

all_fits   = pd.DataFrame(all_rows).sort_values(["Pair", "AIC"]).reset_index(drop=True)
best_fits  = pd.DataFrame(best_rows).sort_values("N", ascending=False).reset_index(drop=True)
best_fits

,Pair,N,Best_Distribution,Parameters,AIC,BIC,LogLik,KS_stat,KS_p,Sample_flag
0,BTW_following_4W,250,weibull_min,"c=2.4049, loc=0.3652, scale=2.1645",622.44,633.01,-308.22,0.0344,0.9187,ok
1,BTW_following_MT_3W,186,weibull_min,"c=1.6347, loc=0.4816, scale=1.5418",440.28,449.95,-217.14,0.0410,0.9001,ok
2,BTW_following_NMT_3W,160,gengamma,"a=0.3835, c=2.8513, loc=0.5315, scale=2.4769",382.63,394.93,-187.31,0.0498,0.8041,ok
3,PR_following_MT_3W,104,genextreme,"c=0.3302, loc=2.5547, scale=0.9291",280.33,288.26,-137.16,0.0706,0.6508,ok
4,BTW_following_2W,73,gengamma,"a=0.4376, c=2.2363, loc=0.6000, scale=2.1461",162.79,171.96,-77.40,0.0713,0.8259,ok
5,PR_following_NMT_3W,59,gengamma,"a=0.1974, c=7.1161, loc=0.4685, scale=3.9221",178.43,186.74,-85.21,0.0581,0.9818,ok
6,PR_following_4W,43,gengamma,"a=0.1499, c=8.7833, loc=0.8197, scale=3.8736",129.53,136.57,-60.76,0.1164,0.5656,caution (small n)
7,PR_following_2W,23,gengamma,"a=0.1402, c=5.9398, loc=1.3333, scale=3.7174",60.70,65.25,-26.35,0.1626,0.5246,caution (small n)


In [5]:
# --- Cell 5: Also show the best fit under BIC (favours simpler models) ---
# Reviewers often want a parsimony check; BIC penalises extra parameters more.
best_bic = (all_fits.sort_values(["Pair", "BIC"])
                    .groupby("Pair", as_index=False).first()
                    [["Pair", "N", "Distribution", "BIC", "AIC", "KS_p"]]
                    .rename(columns={"Distribution": "Best_Distribution_BIC"})
                    .sort_values("N", ascending=False).reset_index(drop=True))
best_bic

,Pair,N,Best_Distribution_BIC,BIC,AIC,KS_p
0,BTW_following_4W,250,weibull_min,633.01,622.44,0.9187
1,BTW_following_MT_3W,186,weibull_min,449.95,440.28,0.9001
2,BTW_following_NMT_3W,160,weibull_min,393.09,383.87,0.8381
3,PR_following_MT_3W,104,genextreme,288.26,280.33,0.6508
4,BTW_following_2W,73,rayleigh,171.81,167.23,0.8130
5,PR_following_NMT_3W,59,rayleigh,184.51,180.36,0.4696
6,PR_following_4W,43,rayleigh,136.25,132.72,0.1201
7,PR_following_2W,23,gengamma,65.25,60.70,0.5246


In [6]:
# --- Cell 6: Save tables to the Tables folder (Excel) ---
out_path = os.path.join(TABLES, "headway_distribution_fits.xlsx")
with pd.ExcelWriter(out_path, engine="openpyxl") as xl:
    best_fits.to_excel(xl, sheet_name="Best_fit_AIC", index=False)
    best_bic.to_excel(xl,  sheet_name="Best_fit_BIC", index=False)
    all_fits.to_excel(xl,  sheet_name="All_fits", index=False)
print("Saved tables:", out_path)

Saved tables: D:\Headway\Tables\headway_distribution_fits.xlsx


In [7]:
# --- Cell 7: Embed a fitted-PDF-vs-histogram chart per pair (matplotlib-free) ---
def sheet_name_for(pair):
    return ("fit_" + pair)[:31].replace("/", "_")

wb = load_workbook(out_path)
for pair, (dist_name, raw, data) in best_store.items():
    dist = getattr(stats, dist_name)
    n = len(data)
    nbins = int(min(20, max(5, round(np.sqrt(n)))))
    dens, edges = np.histogram(data, bins=nbins, density=True)
    mid = (edges[:-1] + edges[1:]) / 2
    xg = np.linspace(data.min(), data.max(), 200)
    yg = dist.pdf(xg, *raw)

    ws = wb.create_sheet(sheet_name_for(pair))
    ws["A1"], ws["B1"] = "hist_mid", "empirical_density"
    ws["D1"], ws["E1"] = "x", f"{dist_name}_pdf"
    for i in range(len(mid)):
        ws.cell(row=i + 2, column=1, value=float(mid[i]))
        ws.cell(row=i + 2, column=2, value=float(dens[i]))
    for i in range(len(xg)):
        ws.cell(row=i + 2, column=4, value=float(xg[i]))
        ws.cell(row=i + 2, column=5, value=float(yg[i]))

    chart = ScatterChart()
    chart.title = f"{pair}  (best fit: {dist_name})"
    chart.x_axis.title = "Time headway (s)"
    chart.y_axis.title = "Density"
    chart.x_axis.delete = False
    chart.y_axis.delete = False

    s_hist = Series(Reference(ws, min_col=2, min_row=1, max_row=len(mid) + 1),
                    Reference(ws, min_col=1, min_row=2, max_row=len(mid) + 1),
                    title_from_data=True)
    s_hist.marker.symbol = "circle"
    s_hist.graphicalProperties.line.noFill = True          # markers only
    s_pdf = Series(Reference(ws, min_col=5, min_row=1, max_row=len(xg) + 1),
                   Reference(ws, min_col=4, min_row=2, max_row=len(xg) + 1),
                   title_from_data=True)
    s_pdf.smooth = True
    chart.series.append(s_hist)
    chart.series.append(s_pdf)
    chart.height, chart.width = 9, 15
    ws.add_chart(chart, "G2")

wb.save(out_path)
print("Embedded per-pair fit charts into:", out_path)

Embedded per-pair fit charts into: D:\Headway\Tables\headway_distribution_fits.xlsx
